In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym
from functools import partial
from collections import deque
import matplotlib.pyplot as plt

from industrial_inventory_env import IndustrialInventoryEnv, generate_student_config

# Configuration
ROLL_NUMBER = "DA25M622"
student_config = generate_student_config(ROLL_NUMBER)
print(f"Student Config for {ROLL_NUMBER}: {student_config}")

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Student Config for DA25M622: {'project_version': 'IITM-6002W-RL-Inventory-2026-v1', 'roll_number': 'DA25M622', 'variant_id': 'V022', 'demand_multiplier_profile': [1.0, 1.1, 0.9], 'initial_inventory_profile': [110, 100, 90], 'lead_time_delay_profile': [0.1, 0.0, 0.05], 'declared_ranges': {'demand_multiplier': [0.85, 1.15], 'initial_inventory': [80, 120], 'lead_time_delay_probability': [0.0, 0.1]}, 'config_fingerprint': '7d77fc79debf206a'}
Using device: cpu


In [2]:
class PolicyNetwork(nn.Module):
    """
    Policy network for the industrial inventory environment.
    Uses a Gaussian policy for continuous action (reorder quantity).
    """
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean_head = nn.Linear(hidden_dim, action_dim)
        # Log standard deviation (shared across states for simplicity)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        mean = torch.tanh(self.mean_head(x)) * 200  # Scale to [-200, 200] reorder range
        std = torch.exp(self.log_std.clamp(-5, 2))
        return mean, std

class ValueNetwork(nn.Module):
    """
    Value network for baseline (critic) in REINFORCE with baseline.
    """
    def __init__(self, state_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.value_head = nn.Linear(hidden_dim, 1)
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        value = self.value_head(x)
        return value.squeeze(-1)

def discounted_returns(rewards, gamma=0.99):
    """
    Calculate discounted returns for a trajectory.
    
    Args:
        rewards: List of rewards [R_1, R_2, ..., R_T]
        gamma: Discount factor
        
    Returns:
        Returns [G_0, G_1, ..., G_{T-1}]
    """
    returns = np.zeros(len(rewards), dtype=np.float32)
    G = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns

def generate_episode(env, policy, seed=None, max_steps=1000):
    """
    Generate one complete episode using the current policy.
    
    Returns:
        Dictionary containing states, actions, rewards, log_probs, and episode info
    """
    state, info = env.reset(seed=seed)
    
    states = []
    actions = []
    rewards = []
    log_probs = []
    
    for step in range(max_steps):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
        
        # Sample action from policy
        mean, std = policy(state_tensor)
        distribution = torch.distributions.Normal(mean, std)
        action = distribution.sample()
        log_prob = distribution.log_prob(action).sum(dim=-1)
        
        # Clip action to valid range (reorder quantity between 0 and 200)
        action_clipped = torch.clamp(action, 0, 200).detach().cpu().numpy()
        
        # Step environment
        next_state, reward, terminated, truncated, info = env.step(action_clipped)
        
        # Store transition
        states.append(state_tensor)
        actions.append(action)
        rewards.append(float(reward))
        log_probs.append(log_prob)
        
        state = next_state
        
        if terminated or truncated:
            # Store final episode info
            episode_info = {
                "states": states,
                "actions": actions,
                "rewards": rewards,
                "log_probs": log_probs,
                "success": bool(terminated),
                "length": len(rewards),
                "total_cost": info.get("costs", {}).get("episode_total", 0)
            }
            return episode_info
    
    # If we hit max_steps without termination
    episode_info = {
        "states": states,
        "actions": actions,
        "rewards": rewards,
        "log_probs": log_probs,
        "success": False,
        "length": len(rewards),
        "total_cost": info.get("costs", {}).get("episode_total", 0)
    }
    return episode_info

def train_reinforce(
    env_fn,
    num_episodes=500,
    gamma=0.99,
    actor_lr=1e-4,
    critic_lr=3e-4,
    use_baseline=True,
    seed=2026,
    eval_interval=50,
    n_eval_episodes=10
):
    """
    Train a REINFORCE agent with optional baseline.
    
    Args:
        env_fn: Function that creates environment
        num_episodes: Number of training episodes
        gamma: Discount factor
        actor_lr: Learning rate for actor (policy)
        critic_lr: Learning rate for critic (value network)
        use_baseline: Whether to use a learned baseline
        seed: Random seed
        eval_interval: Evaluate every N episodes
        n_eval_episodes: Number of episodes for evaluation
    
    Returns:
        policy: Trained policy network
        value_net: Trained value network (if use_baseline)
        history: Training history
    """
    # Set random seed
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # Create environment to get state/action dimensions
    temp_env = env_fn()
    state_dim = temp_env.observation_space.spaces["inventory"].shape[0] + \
                temp_env.observation_space.spaces["demand_history"].shape[0]
    action_dim = temp_env.action_space.shape[0]
    temp_env.close()
    
    # Initialize networks
    policy = PolicyNetwork(state_dim, action_dim).to(DEVICE)
    policy_optimizer = torch.optim.Adam(policy.parameters(), lr=actor_lr)
    
    if use_baseline:
        value_net = ValueNetwork(state_dim).to(DEVICE)
        value_optimizer = torch.optim.Adam(value_net.parameters(), lr=critic_lr)
    else:
        value_net = None
        value_optimizer = None
    
    # Training history
    history = {
        "episode_costs": [],
        "episode_lengths": [],
        "eval_costs": [],
        "eval_lengths": [],
        "actor_losses": [],
        "critic_losses": []
    }
    
    # Learning loop
    for episode_idx in range(num_episodes):
        # Generate episode
        episode_seed = seed + episode_idx if episode_idx == 0 else None
        episode = generate_episode(env_fn(), policy, seed=episode_seed)
        
        # Convert lists to tensors
        states = torch.stack(episode["states"]).to(DEVICE)
        actions = torch.stack(episode["actions"]).to(DEVICE)
        rewards = torch.tensor(episode["rewards"], dtype=torch.float32, device=DEVICE)
        log_probs = torch.stack(episode["log_probs"])
        
        # Calculate returns
        returns = torch.tensor(
            discounted_returns(episode["rewards"], gamma),
            dtype=torch.float32,
            device=DEVICE
        )
        
        # Compute advantages
        if use_baseline:
            with torch.no_grad():
                values = value_net(states)
            advantages = returns - values
            # Normalize advantages for stability
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        else:
            advantages = returns
        
        # Policy (actor) loss
        # L = -sum(gamma^t * A_t * log pi(a_t|s_t))
        gamma_t = gamma ** torch.arange(len(episode["rewards"]), dtype=torch.float32, device=DEVICE)
        actor_loss = -(gamma_t * advantages * log_probs).mean()
        
        # Update actor
        policy_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        policy_optimizer.step()
        
        # Update critic (if using baseline)
        critic_loss = torch.tensor(0.0, device=DEVICE)
        if use_baseline:
            values = value_net(states)
            critic_loss = F.mse_loss(values, returns)
            
            value_optimizer.zero_grad()
            critic_loss.backward()
            torch.nn.utils.clip_grad_norm_(value_net.parameters(), max_norm=1.0)
            value_optimizer.step()
        
        # Store training metrics
        history["episode_costs"].append(episode["total_cost"])
        history["episode_lengths"].append(episode["length"])
        history["actor_losses"].append(actor_loss.item())
        history["critic_losses"].append(critic_loss.item() if use_baseline else 0)
        
        # Evaluation
        if (episode_idx + 1) % eval_interval == 0:
            eval_costs, eval_lengths = evaluate_policy(
                env_fn, policy, n_episodes=n_eval_episodes, seed=seed + episode_idx
            )
            history["eval_costs"].append(np.mean(eval_costs))
            history["eval_lengths"].append(np.mean(eval_lengths))
            
            print(f"Episode {episode_idx + 1:4d} | "
                  f"Cost: {episode['total_cost']:.1f} | "
                  f"Len: {episode['length']:.0f} | "
                  f"Eval Cost: {np.mean(eval_costs):.1f} ± {np.std(eval_costs):.1f}")
    
    return policy, value_net, history

def evaluate_policy(env_fn, policy, n_episodes=20, seed=2026, deterministic=True):
    """
    Evaluate a trained policy.
    
    Args:
        env_fn: Function that creates environment
        policy: Policy network
        n_episodes: Number of episodes for evaluation
        seed: Random seed
        deterministic: If True, use mean action instead of sampling
    
    Returns:
        costs: List of episode costs
        lengths: List of episode lengths
    """
    policy.eval()
    
    costs = []
    lengths = []
    
    with torch.no_grad():
        for ep in range(n_episodes):
            env = env_fn()
            state, info = env.reset(seed=seed + ep)
            
            total_cost = 0
            episode_len = 0
            
            while True:
                state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
                
                if deterministic:
                    # Use mean action (deterministic)
                    mean, _ = policy(state_tensor)
                    action = torch.clamp(mean, 0, 200).cpu().numpy()
                else:
                    # Sample from policy
                    mean, std = policy(state_tensor)
                    distribution = torch.distributions.Normal(mean, std)
                    action = torch.clamp(distribution.sample(), 0, 200).cpu().numpy()
                
                state, reward, terminated, truncated, info = env.step(action)
                episode_len += 1
                
                if terminated or truncated:
                    total_cost = info.get("costs", {}).get("episode_total", 0)
                    break
            
            costs.append(total_cost)
            lengths.append(episode_len)
            env.close()
    
    policy.train()
    return costs, lengths

def plot_training_history(history, title="REINFORCE Training Progress"):
    """
    Plot training history.
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Episode costs
    axes[0, 0].plot(history["episode_costs"], alpha=0.5, label="Episode Cost")
    if history["eval_costs"]:
        axes[0, 0].plot(
            np.linspace(0, len(history["episode_costs"]), len(history["eval_costs"])),
            history["eval_costs"],
            'r-', linewidth=2, label="Evaluation Cost"
        )
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Total Cost")
    axes[0, 0].set_title("Episode Costs")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Episode lengths
    axes[0, 1].plot(history["episode_lengths"], alpha=0.5, label="Episode Length")
    if history["eval_lengths"]:
        axes[0, 1].plot(
            np.linspace(0, len(history["episode_lengths"]), len(history["eval_lengths"])),
            history["eval_lengths"],
            'r-', linewidth=2, label="Eval Length"
        )
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].set_ylabel("Length")
    axes[0, 1].set_title("Episode Lengths")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # Actor losses
    axes[1, 0].plot(history["actor_losses"])
    axes[1, 0].set_xlabel("Episode")
    axes[1, 0].set_ylabel("Actor Loss")
    axes[1, 0].set_title("Actor Loss")
    axes[1, 0].grid(alpha=0.3)
    
    # Critic losses (if available)
    if any(history["critic_losses"]):
        axes[1, 1].plot(history["critic_losses"])
        axes[1, 1].set_xlabel("Episode")
        axes[1, 1].set_ylabel("Critic Loss")
        axes[1, 1].set_title("Critic Loss")
    else:
        axes[1, 1].text(0.5, 0.5, "No baseline used", 
                       horizontalalignment='center', verticalalignment='center')
        axes[1, 1].set_title("Critic Loss (No baseline)")
    axes[1, 1].grid(alpha=0.3)
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [3]:
# Create environment function
def make_env():
    return IndustrialInventoryEnv(
        student_config=student_config,
        scenario_mode="random",
        domain_randomization=True
    )

# Train REINFORCE with baseline
print("Training REINFORCE with baseline...")
policy, value_net, history = train_reinforce(
    env_fn=make_env,
    num_episodes=500,
    gamma=0.99,
    actor_lr=3e-4,
    critic_lr=6e-4,
    use_baseline=True,
    seed=2026,
    eval_interval=50,
    n_eval_episodes=15
)

Training REINFORCE with baseline...


TypeError: must be real number, not dict

In [ ]:
# Plot training history
plot_training_history(history)

In [ ]:
# Final evaluation
print("\nFinal Evaluation:")
final_costs, final_lengths = evaluate_policy(
    make_env, policy, n_episodes=30, seed=999, deterministic=True
)

print(f"Mean Cost: {np.mean(final_costs):.2f} ± {np.std(final_costs):.2f}")
print(f"Mean Length: {np.mean(final_lengths):.2f} ± {np.std(final_lengths):.2f}")
print(f"Cost Range: [{np.min(final_costs):.2f}, {np.max(final_costs):.2f}]")

# Compare with default policy (random actions)
print("\nComparing with random policy:")
random_costs, _ = evaluate_policy(
    make_env, policy, n_episodes=30, seed=888, deterministic=False
)
print(f"Random policy mean cost: {np.mean(random_costs):.2f} ± {np.std(random_costs):.2f}")
print(f"Learned policy mean cost: {np.mean(final_costs):.2f} ± {np.std(final_costs):.2f}")
print(f"Improvement: {np.mean(random_costs) - np.mean(final_costs):.2f}")